# Teste do `climasus_readdbc_py` com arquivos reais do FTP DATASUS

Objetivo: baixar arquivos `.dbc` reais de `ftp://ftp.datasus.gov.br/dissemin/publicos`, ler com o leitor puro-Python `climasus_readdbc_py` e registrar qualquer erro de download, descompressão ou parsing DBF.

Este notebook não usa `pysus`, `pyreaddbc` nem binários externos. Ele testa diretamente o pacote `climasus_readdbc_py`.

## 1. Preparar ambiente

Execute esta célula a partir do repositório `climasus_readdbc_py` ou da pasta `notebooks/`. Se o pacote ainda não estiver importável, a célula instala o projeto local em modo editável.

In [ ]:
from __future__ import annotations

import hashlib
import platform
import struct
import subprocess
import sys
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        f"Nao encontrei pyproject.toml em {PROJECT_ROOT}. "
        "Rode o notebook dentro de climasus_readdbc_py ou notebooks/."
    )

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import climasus_readdbc_py as readdbc
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)])
    import climasus_readdbc_py as readdbc

CACHE_DIR = PROJECT_ROOT / "dados" / "ftp_datasus_samples"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Python", sys.version)
print("Platform", platform.platform())
print("Project root", PROJECT_ROOT)
print("Cache", CACHE_DIR)
print("climasus_readdbc_py", readdbc.__version__)

## 2. Amostras reais do FTP

As amostras abaixo usam UF `RR` para manter os downloads pequenos. Cada entrada pode ter mais de uma URL candidata, porque alguns diretórios do DATASUS variam entre `PRELIM`, `CID10`, `NOV` e séries mensais.

In [ ]:
FTP_ROOT = "ftp://ftp.datasus.gov.br/dissemin/publicos"

SAMPLES = [
    {
        "label": "SIM-DO RR 2023",
        "system": "SIM-DO",
        "urls": [
            f"{FTP_ROOT}/SIM/PRELIM/DORES/DORR2023.dbc",
            f"{FTP_ROOT}/SIM/CID10/DORES/DORR2023.dbc",
        ],
    },
    {
        "label": "SINASC RR 2022",
        "system": "SINASC",
        "urls": [
            f"{FTP_ROOT}/SINASC/PRELIM/DNRES/DNRR2022.dbc",
            f"{FTP_ROOT}/SINASC/NOV/DNRES/DNRR2022.dbc",
        ],
    },
    {
        "label": "SIH-RD RR 2023-01",
        "system": "SIH-RD",
        "urls": [
            f"{FTP_ROOT}/SIHSUS/200801_/Dados/RDRR2301.dbc",
        ],
    },
]

SAMPLES

## 3. Funções auxiliares de download e diagnóstico

In [ ]:
@dataclass
class SampleResult:
    label: str
    system: str
    url: str | None
    path: str | None
    status: str
    file_size: int | None = None
    md5: str | None = None
    header_size: int | None = None
    record_size: int | None = None
    n_records_header: int | None = None
    rows: int | None = None
    columns: int | None = None
    seconds_download: float | None = None
    seconds_read: float | None = None
    error_type: str | None = None
    error: str | None = None


def local_name_from_url(url: str) -> str:
    return url.rsplit("/", 1)[-1]


def inspect_dbc_header(raw: bytes) -> dict[str, int | str]:
    if len(raw) < 32:
        return {"error": "arquivo menor que 32 bytes"}
    return {
        "version_byte": raw[0],
        "n_records_header": struct.unpack_from("<I", raw, 4)[0],
        "header_size": struct.unpack_from("<H", raw, 8)[0],
        "record_size": struct.unpack_from("<H", raw, 10)[0],
        "first_16_bytes_hex": raw[:16].hex(" "),
    }


def download_first_available(
    urls: list[str],
    cache_dir: Path,
    timeout: int = 120,
) -> tuple[str, Path, float]:
    errors: list[str] = []
    for url in urls:
        path = cache_dir / local_name_from_url(url)
        if path.exists() and path.stat().st_size > 0:
            return url, path, 0.0
        start = time.perf_counter()
        try:
            with urlopen(url, timeout=timeout) as response:
                data = response.read()
            path.write_bytes(data)
            return url, path, time.perf_counter() - start
        except Exception as exc:
            errors.append(f"{url}: {type(exc).__name__}: {exc}")
    raise RuntimeError("Nenhuma URL candidata baixou com sucesso:\n" + "\n".join(errors))


def run_sample(sample: dict) -> tuple[SampleResult, pd.DataFrame | None]:
    label = sample["label"]
    system = sample["system"]
    try:
        url, path, seconds_download = download_first_available(sample["urls"], CACHE_DIR)
        raw = path.read_bytes()
        header = inspect_dbc_header(raw)
        start = time.perf_counter()
        df = readdbc.read_dbc(path)
        seconds_read = time.perf_counter() - start
        result = SampleResult(
            label=label,
            system=system,
            url=url,
            path=str(path),
            status="ok",
            file_size=path.stat().st_size,
            md5=hashlib.md5(raw).hexdigest(),
            header_size=header.get("header_size"),
            record_size=header.get("record_size"),
            n_records_header=header.get("n_records_header"),
            rows=len(df),
            columns=len(df.columns),
            seconds_download=seconds_download,
            seconds_read=seconds_read,
        )
        return result, df
    except Exception as exc:
        result = SampleResult(
            label=label,
            system=system,
            url=None,
            path=None,
            status="error",
            error_type=type(exc).__name__,
            error=str(exc),
        )
        traceback.print_exc(limit=5)
        return result, None

## 4. Rodar bateria básica

Critério mínimo: todos os arquivos baixados devem produzir DataFrame com pelo menos uma linha e uma coluna.

In [ ]:
results: list[SampleResult] = []
frames: dict[str, pd.DataFrame] = {}

for sample in SAMPLES:
    print(f"\n=== {sample['label']} ===")
    result, df = run_sample(sample)
    results.append(result)
    print(result)
    if df is not None:
        frames[result.label] = df
        print(df.shape)
        display(df.head(3))

summary = pd.DataFrame([r.__dict__ for r in results])
summary

## 5. Validar DBC → DBF → DataFrame

Esta célula testa o caminho interno explicitamente: `dbc_to_dbf(bytes)` seguido de `read_dbf(bytes)`. A forma direta `read_dbc(path)` deve ter a mesma dimensão.

In [ ]:
roundtrip_rows = []

for result in results:
    if result.status != "ok" or result.path is None:
        continue
    raw = Path(result.path).read_bytes()
    dbf_bytes = readdbc.dbc_to_dbf(raw)
    df_dbf = readdbc.read_dbf(dbf_bytes)
    df_dbc = frames[result.label]
    same_shape = df_dbf.shape == df_dbc.shape
    roundtrip_rows.append({
        "label": result.label,
        "dbc_shape": df_dbc.shape,
        "dbf_shape": df_dbf.shape,
        "same_shape": same_shape,
        "dbf_bytes": len(dbf_bytes),
    })
    assert same_shape, (
        f"Shape diferente no roundtrip para {result.label}: "
        f"{df_dbc.shape} vs {df_dbf.shape}"
    )

pd.DataFrame(roundtrip_rows)

## 6. Falhar explicitamente se houver erro

Use esta célula quando quiser transformar o notebook em teste manual: se alguma amostra falhou, ela levanta `AssertionError` com a tabela de erros.

In [ ]:
errors = summary[summary["status"] != "ok"]
if not errors.empty:
    display(errors[["label", "system", "error_type", "error"]])
    raise AssertionError(f"{len(errors)} amostra(s) falharam")

assert (summary["rows"] > 0).all(), "Alguma amostra retornou zero linhas"
assert (summary["columns"] > 0).all(), "Alguma amostra retornou zero colunas"
print("Todas as amostras básicas passaram.")

## 7. Teste opcional de estresse

Ative `RUN_STRESS = True` para testar arquivos maiores, como SP/RJ. Mantenha desligado no uso normal para evitar downloads grandes.

In [ ]:
RUN_STRESS = False

STRESS_SAMPLES = [
    {
        "label": "SIM-DO SP 2023",
        "system": "SIM-DO",
        "urls": [
            f"{FTP_ROOT}/SIM/PRELIM/DORES/DOSP2023.dbc",
            f"{FTP_ROOT}/SIM/CID10/DORES/DOSP2023.dbc",
        ],
    },
    {
        "label": "SIH-RD SP 2023-01",
        "system": "SIH-RD",
        "urls": [
            f"{FTP_ROOT}/SIHSUS/200801_/Dados/RDSP2301.dbc",
        ],
    },
]

if RUN_STRESS:
    stress_results = []
    for sample in STRESS_SAMPLES:
        result, df = run_sample(sample)
        stress_results.append(result)
        print(result)
        if df is not None:
            print(df.shape)
            display(df.head(3))
    pd.DataFrame([r.__dict__ for r in stress_results])
else:
    print("Stress desativado. Altere RUN_STRESS para True para rodar.")